In [1]:
import os
import glob
import sys

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr5"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
    cell_types = ["c1", "c2", "c4", "c6", "c17"]
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
    cell_types = ["c1", "c2", "c4", "c6", "c13", "c17"]
else:
    raise ValueError

In [5]:
PATH_FROM = f'../../data/lib2/{utr_type.upper()}_zinb_norm_2024-06-04.csv'
df_src = pd.read_csv(PATH_FROM)

In [6]:
df_src["seq"] = df_src["seq"].str.upper()

In [7]:
mc_data = df_src.groupby(["seq", "cell_type"])[["1", "2", "3", "4"]].sum().reset_index()
mc_data

,seq,cell_type,1,2,3,4
0,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c1,614.0,395.0,685.0,815.0
1,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c17,3442.0,3260.0,2909.0,4201.0
2,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c2,3384.0,2467.0,1713.0,1320.0
3,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c4,3926.0,1645.0,1375.0,933.0
4,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c6,4475.0,2894.0,3129.0,2018.0
...,...,...,...,...,...,...
53894,TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCG...,c4,15230.0,14372.0,13533.0,13944.0
53895,TTTTTTGTGTGCGTGCACGTCCTCGTTATCGTGCATCCATGGCGCG...,c6,10436.0,3150.0,4367.0,6943.0
53896,TTTTTTTTTTGAGATTTGAGATTTTTGTGTTTGGTTTGTAAGTTGT...,c17,158.0,171.0,329.0,355.0
53897,TTTTTTTTTTGAGATTTGAGATTTTTGTGTTTGGTTTGTAAGTTGT...,c2,218.0,75.0,148.0,197.0


In [8]:
mc_data["mass_center"] = (mc_data[["1", "2", "3", "4"]] * np.arange(1, 5)).sum(axis=1) / mc_data[["1", "2", "3", "4"]].sum(axis=1)

In [9]:
num_classes = len(cell_types)

In [10]:
from itertools import product

df = pd.DataFrame(product(df_src["seq"].unique(), cell_types), columns=["seq", "cell_type"])
df.head()

,seq,cell_type
0,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c1
1,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c2
2,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c4
3,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c6
4,AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGG...,c17


In [11]:
df = pd.merge(df, mc_data[["seq", "cell_type", "mass_center"]], on=["seq", "cell_type"], how="left", validate="1:1")

In [12]:
batch_size = 1024

In [13]:
num_workers = 32

In [14]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [15]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [16]:
ckpt = torch.load(MODEL_PATH)
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding

loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

<All keys matched successfully>

In [17]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[0],
    deterministic=True,
    # gradient_clip_val=1e-5,
    # gradient_clip_algorithm="norm",
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [18]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [19]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")
seq_embedding

array([[0.1827774 , 0.10244783, 0.06879551, ..., 0.15200016, 0.19421025,
        0.18063432],
       [0.1414008 , 0.1465124 , 0.11121053, ..., 0.10293644, 0.16323526,
        0.13787374],
       [0.134791  , 0.14385624, 0.13766022, ..., 0.11820646, 0.17919724,
        0.19467232],
       ...,
       [0.1593156 , 0.18433043, 0.11291756, ..., 0.16081482, 0.1711383 ,
        0.1554606 ],
       [0.14476472, 0.14328142, 0.10294867, ..., 0.12335016, 0.20677307,
        0.13151571],
       [0.136452  , 0.15149586, 0.09884147, ..., 0.13544111, 0.19257429,
        0.0680006 ]], dtype=float32)

In [20]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)
pred_mass_center

array([[2.235352 , 2.1723752, 2.3283   , 2.3360715, 2.3744662],
       [2.3077614, 2.412075 , 2.4586263, 2.3660872, 2.4087706],
       [2.419005 , 2.4964237, 2.545624 , 2.5203745, 2.5498252],
       ...,
       [2.6001732, 2.5907695, 2.5937593, 2.6015468, 2.5947545],
       [2.4154246, 2.512764 , 2.4015381, 2.3853805, 2.4133708],
       [2.3504498, 2.5131521, 2.4192805, 2.2934577, 2.3176143]],
      dtype=float32)

In [21]:
result = df.pivot(columns="cell_type", index="seq", values="mass_center")
result.columns = pd.MultiIndex.from_product([["mass_center"], result.columns])
result.head()

mass_center            \
cell_type                                                   c1       c17   
seq                                                                        
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC    2.677959  2.569722   
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC    2.591365  2.374306   
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC    2.419520  2.148036   
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG    2.676759  2.772232   
AAAAACAGCCGAGGCCACCGGGCAGGACTGCAGTGTTCTTTCTAAATATA         NaN       NaN   

                                                                        \
cell_type                                                 c2        c4   
seq                                                                      
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC  2.109072  1.913060   
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC  2.794696  2.845000   
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC  1.888015  2.171457   
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG  2.606655  2.502079   
AAAAACAGCCGAGGCCACCGGGCAGGACTGCAGTGTTCTTTCTAAATATA       NaN  2.593232   

                                                              
cell_type                                                 c6  
seq                                                           
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC  2.214925  
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC  2.470830  
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC  2.404858  
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG  2.800523  
AAAAACAGCCGAGGCCACCGGGCAGGACTGCAGTGTTCTTTCTAAATATA       NaN

In [22]:
result[
    pd.MultiIndex.from_product([["pred_mass_center"], result["mass_center"].columns])
] = pred_mass_center
result["pred_mass_center"].head()

cell_type,c1,c17,c2,c4,c6
seq,,,,,
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC,2.235352,2.172375,2.328300,2.336071,2.374466
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC,2.307761,2.412075,2.458626,2.366087,2.408771
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC,2.419005,2.496424,2.545624,2.520375,2.549825
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG,2.466333,2.455868,2.535168,2.483383,2.519254
AAAAACAGCCGAGGCCACCGGGCAGGACTGCAGTGTTCTTTCTAAATATA,2.158932,2.416485,2.232741,2.022552,1.974816


In [23]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [24]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c6_022,c6_023,c6_024,c6_025,c6_026,c6_027,c6_028,c6_029,c6_030,c6_031
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC,0.182777,0.102448,0.068796,0.174596,0.073827,0.100016,0.125878,0.096271,0.084169,0.074872,...,-0.023058,0.124752,0.150029,0.143469,0.241267,-0.057914,0.145602,0.152000,0.194210,0.180634
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC,0.141401,0.146512,0.111211,0.134049,0.153072,0.103950,0.123336,0.138253,0.124936,0.144997,...,0.023018,0.142749,0.130637,0.143916,0.150199,0.003270,0.163896,0.102936,0.163235,0.137874
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC,0.134791,0.143856,0.137660,0.176925,0.105589,0.133643,0.170833,0.090401,0.105071,0.114902,...,0.103910,0.140764,0.119623,0.163092,0.126567,-0.056239,0.164292,0.118206,0.179197,0.194672
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG,0.136578,0.190032,0.083655,0.168899,0.133308,0.101772,0.115243,0.136464,0.137851,0.150299,...,0.078720,0.128658,0.129674,0.098766,0.180798,-0.002088,0.154702,0.084946,0.183337,0.180277
AAAAACAGCCGAGGCCACCGGGCAGGACTGCAGTGTTCTTTCTAAATATA,0.131528,0.091646,0.133701,0.102706,0.158316,0.100434,0.096783,0.138023,0.130565,0.131887,...,-0.131695,0.125342,0.100398,0.171503,0.111396,-0.060069,0.193236,0.059194,0.137497,0.073444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTGCGTGTGCGCTGTGACTGGTGCTCCGTTTTCCTCGCTCATGGTGG,0.164786,0.191665,0.149073,0.189911,0.161712,0.124536,0.122063,0.220222,0.139372,0.129565,...,0.053871,0.127217,0.105246,0.069263,0.171797,0.100032,0.125496,0.120328,0.185239,0.177281
TTTTTTCCTTTTCCTTTTCCTCCCCCCCTCCTTTTCCATTATTTTTTAAT,0.164519,0.147968,0.147326,0.174774,0.127871,0.148125,0.153580,0.124589,0.141108,0.117552,...,0.074104,0.144095,0.140165,0.147503,0.113678,0.111440,0.144859,0.145189,0.133817,0.133598
TTTTTTCTGCTGCAGGCGGAGCGGCGCGCGGAGGAGGCGCCGGGAGCCGC,0.159316,0.184330,0.112918,0.170566,0.152236,0.168309,0.140380,0.167897,0.151845,0.148726,...,0.103480,0.140929,0.155253,0.120936,0.153283,0.150965,0.151290,0.160815,0.171138,0.155461


### Counting k-mers

In [25]:
sys.path.append("../../benchmark/kmers")
from kmer_model import KmerLinearRegressor

In [26]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [27]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAATGTGACGGCGGCGGCGGCGGCGGCGGCGGCAGCGGCGGCGGC,10,13,25,2,7,1,1,1,1,0,...,0,0,1,0,0,1,0,0,0,0
AAAAAATGGAGCGTGCGGCCTTGGCTGGTGGAGGAGGCGCAGGCAGCCCC,11,12,21,6,5,0,5,1,2,4,...,0,0,0,1,4,0,0,0,1,0
AAAAAATGGAGGCGGAGCTGCGGCGGCGGCGGCGGCTGTGGCGGGACGTC,9,11,25,5,5,1,2,1,0,0,...,0,0,0,1,2,1,0,0,0,0
AAAAAATGGCAGCGAGGGCGTTGGCGACGGCTGAGAGCGGAGCCGGCGTG,12,10,23,5,5,1,5,1,1,1,...,0,0,1,0,2,0,0,0,1,0
AAAAACAGCCGAGGCCACCGGGCAGGACTGCAGTGTTCTTTCTAAATATA,16,12,12,10,6,3,4,2,4,3,...,0,2,0,1,0,1,0,2,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTGCGTGTGCGCTGTGACTGGTGCTCCGTTTTCCTCGCTCATGGTGG,2,12,16,20,0,1,0,1,1,2,...,1,0,1,3,3,2,0,1,1,5
TTTTTTCCTTTTCCTTTTCCTCCCCCCCTCCTTTTCCATTATTTTTTAAT,4,17,0,29,1,0,0,3,1,11,...,0,0,0,0,0,0,2,4,0,14
TTTTTTCTGCTGCAGGCGGAGCGGCGCGCGGAGGAGGCGCCGGGAGCCGC,5,14,23,8,0,0,5,0,1,2,...,0,1,0,2,0,0,0,1,0,4


In [28]:
result.to_parquet(f"embeddings_mapperless_{utr_type}_library2.parquet")

---